In [1]:
import torch
import torchaudio
from openunmix import predict



In [11]:
import os
import subprocess
import whisper
from pathlib import Path
import shutil

if shutil.which("ffmpeg") is None:
    winget_packages = Path(os.environ.get("LOCALAPPDATA", "")) / "Microsoft" / "WinGet" / "Packages"
    ffmpeg_bin = next(
        (
            directory
            for directory in winget_packages.glob("Gyan.FFmpeg.Shared_*/ffmpeg-*/bin")
            if (directory / "ffmpeg.exe").is_file()
        ),
        None,
    )
    if ffmpeg_bin is None:
        raise FileNotFoundError(
            "FFmpeg was not found. Install it with: winget install --id Gyan.FFmpeg.Shared"
        )
    os.environ["PATH"] = str(ffmpeg_bin) + os.pathsep + os.environ.get("PATH", "")

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


In [14]:

# 1. Cấu hình tên file đầu vào
input_file = "Đen - Trốn Tìm ft. MTV band (M-V).mp3"  # Thay bằng đường dẫn file nhạc của bạn

print("--- Đang tiến hành tách giọng hát (Demucs) ---")
# Sử dụng mô hình htdemucs để tách stem (tách vocal ra khỏi nhạc nền)
# Lệnh này sẽ tạo ra thư mục separated/htdemucs/input_file_name/
subprocess.run(["demucs", "-n", "htdemucs", input_file], check=True)

# Xác định đường dẫn file vocal sau khi tách
base_name = os.path.splitext(os.path.basename(input_file))[0]
vocal_path = os.path.join("separated", "htdemucs", base_name, "vocals.wav")

print(f"Đã tách xong! File giọng hát lưu tại: {vocal_path}")

print("--- Đang chuyển giọng hát thành văn bản (Whisper) ---")
# Tải mô hình Whisper (bạn có thể đổi thành 'small', 'medium', hoặc 'large' tùy cấu hình)
model = whisper.load_model("base")

# Thực hiện nhận diện giọng nói (có thể chỉ định language="vi" nếu là tiếng Việt)
result = model.transcribe(vocal_path, language="vi")

# In kết quả ra màn hình
print("\n--- KẾT QUẢ VĂN BẢN ---")
print(result["text"])

# Lưu kết quả ra file txt
output_txt = f"{base_name}_transcript.txt"
with open(output_txt, "w", encoding="utf-8") as f:
    f.write(result["text"])

print(f"\nĐã lưu toàn bộ văn bản vào file: {output_txt}")

--- Đang tiến hành tách giọng hát (Demucs) ---
Đã tách xong! File giọng hát lưu tại: separated\htdemucs\Đen - Trốn Tìm ft. MTV band (M-V)\vocals.wav
--- Đang chuyển giọng hát thành văn bản (Whisper) ---


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



--- KẾT QUẢ VĂN BẢN ---
 Chuyên những tán kê nơi rồi nhưng đoá hoa ha Có hai đứa nhốt đang chơi trốn tiền Điêm ngoài tiền mái nên quen lỗi Anh đi vì thì em chốn, anh đi chốn em không tìm Làm em không gần xóm của cốm anh mắt công gì Nếu mà có kết bố anh sẽ nạn chắc bỏ công Vì nhớ những nhiêu đặc quán giở nó khô thần về tầm Chúng ta rồi sẽ có, có những chuyến đi dài Và từ hợp lấy mà thứa vì trong tình trường làm biến ghi bài Cô chuyện của chúng ta bỏ mức có thêm một minh bi hai Vì cách mà em gọi anh không có u và thiếu y dài Vì ra tàn phố người ta đã bật đèn đường Hướng quen sư là mang tay thật thêm thuồn Nỗi buồn vàng được cứ như là chứa đồng thao hết Như là biết, không phải em chỉ dọi cửa lòng nhau Hế cửa sora má xem Có một trang thí sĩ đến nghe nhà em Vết nhật tình mạch ngọt, tựa như cây ca rem Anh ta sẽ đứng, anh ở đây ca rem Dạ Chên nhân tăng kê nữa rồi những đó hòa Có hai đưa nhóc đang chơi trốn tì Tìm hòa thì máy nên quên lỗi Hồi đó anh dục đề như đang kê mắc cỡ Gửa em em em măn 

In [15]:
print(result["text"])

 Chuyên những tán kê nơi rồi nhưng đoá hoa ha Có hai đứa nhốt đang chơi trốn tiền Điêm ngoài tiền mái nên quen lỗi Anh đi vì thì em chốn, anh đi chốn em không tìm Làm em không gần xóm của cốm anh mắt công gì Nếu mà có kết bố anh sẽ nạn chắc bỏ công Vì nhớ những nhiêu đặc quán giở nó khô thần về tầm Chúng ta rồi sẽ có, có những chuyến đi dài Và từ hợp lấy mà thứa vì trong tình trường làm biến ghi bài Cô chuyện của chúng ta bỏ mức có thêm một minh bi hai Vì cách mà em gọi anh không có u và thiếu y dài Vì ra tàn phố người ta đã bật đèn đường Hướng quen sư là mang tay thật thêm thuồn Nỗi buồn vàng được cứ như là chứa đồng thao hết Như là biết, không phải em chỉ dọi cửa lòng nhau Hế cửa sora má xem Có một trang thí sĩ đến nghe nhà em Vết nhật tình mạch ngọt, tựa như cây ca rem Anh ta sẽ đứng, anh ở đây ca rem Dạ Chên nhân tăng kê nữa rồi những đó hòa Có hai đưa nhóc đang chơi trốn tì Tìm hòa thì máy nên quên lỗi Hồi đó anh dục đề như đang kê mắc cỡ Gửa em em em măn nền hợp tới mức gây tắc t

In [ ]:
tex = """ Chuyến những tán kê nơi rồi nhưng đoá hoa ha Có hai đứa nhốt đang chơi chốn tìm Điêm hoài tiền mái nên quen lỗi Anh đi tìm thì em chốn, anh đi chốn em không tìm Làm em không gần xóm của cốm anh mắt công một gì Nếu mà có kết bố anh sẽ nạn chắc bỏ công Vì nhớ những nhiêu đặc quán giở nó khô thần về tầm Chúng ta rồi sẽ có, có những chuyến đi dài Và từ hợp lấy mà thứa vì trong tình trường làm biến ghi bài Cô chuyện của chúng ta bỏ có thể một minh bi hai vì Các mà em gọi anh không có u và thiếu y dài Ví xa tàn phố người ta đã bật đèn đường Hướng quen xưa là mang tay thật thêm thuồn Nỗi buồn vàng được cứ như là chứa đồng, thau hết Như là biết, không phải em chỉ dọi cửa lòng nhau Hế, cửa số ra má xem Có một trang thị sĩ, đứng ở ngay nhà em Vết nhạc tình mạc ngọt, tựa như cây ca rem Anh ta sẽ đứng ở nơi đây ca rem Dạ Dễ những tăng kê nơ rồi những đó hoa Có hai đưa nhóc đang trời trốn tì Tìm hoài thì máy nên quên lỗi đêm Hồi đó em được rèi với đăng kê mắt cỡ Gừng em làm mắt này hợp tới mức gây tắc thở Ta đều không biết có điều gì xa đắm mì xác nhở Cho chơi chốn tìm ngày đó sau này đầy trắc chở Ta săng bắn những khác vòng và hai lượng những dất mơ Ta giây ở trong cái ước mộng thức mà lấy đi nhiều Thì giờ ta đào những cái hố mà không biết có ngày bị lọ Để rất lâu sau này chúng ta có mày nhận ra Không phải tất cả bông hoa thì đều sẽ có những nguyện ngọ Gở ta em làm anh mất dọc hoài Hẹt như là một sức mộng giài Ta đi tìm cả thế giới nhưng mà là chốn nhau Bức về thằng ý chơi từ lúc mới chớp đau Người đi tìm, cây người đi chốn Người đi chốn, cây người đi tìm Tình yêu từ những ngày sưa đã Mục quen dưới tăng cây em liền Đó ba cây cây trong một cây vừa Chết những tăng cây đơ rồi những đá mua Có hai đưa nhóc tăng chơi chốn tìm Dì ngoài chỉ mái lên quên loại về Ai đó đã chắc hết cây và cùng chẳng thay vừa Chắc hết mạnh phúc, ít vui vơi mạnh phúc Nhà em con chói con zambar Nhìn cố đăng cuối những người chương thay Là khi muốn chốn nhưng không ai tìm Như khi ta muốn ta được bé lại Để khi đi chốn con người đi tìm Như khi ta muốn ta được bé lại Để khi đi chốn con người đi tìm"""


In [19]:
from google import genai

In [ ]:

# Gán API key vào biến môi trường hoặc cấu hình trực tiếp
os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6Lm4BzCYKsXRsn7UQmIOpQVpZ-ZbDO4-JmZuu2zUSJ9fg"

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[{"role": "user", "content": result["text"]}, {"role": "assistant", "content": "Hãy tóm tắt nội dung văn bản trên thành 3 câu ngắn gọn."}]
)

print(response.text)

ValidationError: 33 validation errors for _GenerateContentParameters
contents.Content
  Input should be a valid dictionary or object to extract fields from [type=model_attributes_type, input_value=[{'role': 'user', 'conten... 3 câu ngắn gọn.'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_attributes_type
contents.str
  Input should be a valid string [type=string_type, input_value=[{'role': 'user', 'conten... 3 câu ngắn gọn.'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
contents.is-instance[Image]
  Input should be an instance of Image [type=is_instance_of, input_value=[{'role': 'user', 'conten... 3 câu ngắn gọn.'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/is_instance_of
contents.File
  Input should be a valid dictionary or object to extract fields from [type=model_attributes_type, input_value=[{'role': 'user', 'conten... 3 câu ngắn gọn.'}

In [12]:
import splittt 
import sumar

In [5]:
splitter = splittt.splitt("input_file_name=Đen - Trốn Tìm ft. MTV band (M-V).mp3")

--- Đang tiến hành tách giọng hát (Demucs) ---
Đã tách xong! File giọng hát lưu tại: separated\htdemucs\Đen - Trốn Tìm ft. MTV band (M-V)\vocals.wav
--- Đang chuyển giọng hát thành văn bản (Whisper) ---


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [7]:
print(splitter)

 Chuyến những tán kê nơi rồi nhưng đoá hoa ha Có hai đứa nhốt đang chơi trốn tiền Điêm hoài tiền mái nên quen lỗi Anh đi tìm thì em chốn, anh đi chốn em không tìm Làm em không gần xóm của cốm mắn mắt công một gì Nếu mà có kết bố anh sẽ nạn chắc bỏ công Vì nhớ những nhiêu đặc quán giở nó khô thần về tầm Chúng ta rồi sẽ có, có những chuyến đi dài Và từ hợp lấy mà thứa vì trong tình trường làm biến ghi bài Cô chuyện của chúng ta bỏng có thể một minh bi hai vì Các mẹ em gọi anh không có u và thứ y dài Ví xa tàn phố người ta đã bật đèn đường Hướng quen xưa là mắn tay thật thêm thuồng Nỗi buồn vèng được cứ như là chứa đồng Tao hết như là biết, không phải em chỉ dọi cửa lòng nhau Hế cửa sora má xem Có một trang thị sĩ đến ở ngay nhà em Vết nhạc tình mạc ngọt tựa như cây ca rem Anh ta sẽ đứng ở nơi đây ca rem Trời ơi chung một cây vừa Dạ Chên nhân tăng kê nơi rồi những đoá hoa Cô hai đưa nhóc đang trời trốn tìa Tìm hoài tìm máy nên quên lâu Hồi đó anh được nền với đăng kê mắc cỡ Vững em mắn mắ

In [13]:
res = sumar.generate_text(splitter)

ValidationError: 33 validation errors for _GenerateContentParameters
contents.Content
  Input should be a valid dictionary or object to extract fields from [type=model_attributes_type, input_value=[{'role': 'user', 'conten...bằng tiếng Việt'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_attributes_type
contents.str
  Input should be a valid string [type=string_type, input_value=[{'role': 'user', 'conten...bằng tiếng Việt'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
contents.is-instance[Image]
  Input should be an instance of Image [type=is_instance_of, input_value=[{'role': 'user', 'conten...bằng tiếng Việt'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/is_instance_of
contents.File
  Input should be a valid dictionary or object to extract fields from [type=model_attributes_type, input_value=[{'role': 'user', 'conten...bằng tiếng Việt'}], i